In [1]:
import numpy as np
import pandas as pd

# Dataset & DataLoader

In [2]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T

from dataset import TrainDataset, TestDataset

image_size = 64
batch_size = 128
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_dataset = TrainDataset(root_path = './cs441-assn3-data/Train_64/', transform = train_transform)
test_dataset = TestDataset(root_path = './cs441-assn3-data/Test_64/', transform = eval_transform)

val_ratio = 0.2 # train 80%, val 20%

# 전체 길이 기준으로 train/val 길이 계산
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset_split, val_dataset = random_split(
    train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(dataset=train_dataset_split,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last = True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

# Your Awesome Model

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.8.0+cu129
True


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=15):
        super(ResNet, self).__init__()
        self.in_planes = 64

        # [수정됨] 64x64 이미지용 Stem
        # 표준 ResNet의 7x7 Conv + MaxPool 대신 3x3 Conv를 사용하여 정보를 보존합니다.
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        # ResNet-34 Layers
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        # self.maxpool(out) # 64x64에서는 MaxPool 생략
        
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        
        # Global Average Pooling
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def ResNet34(num_classes=15):
    # ResNet-34 configuration: [3, 4, 6, 3] blocks
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes=num_classes)

# 모델 생성 (device는 노트북 상단에 정의되어 있어야 함)
model = ResNet34(num_classes=15).to(device)

# 파라미터 수 확인
num_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-34 Parameters: {num_params/1e6:.2f} M")

# Model parameter checking

In [6]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 49295887
Parameter usage : 49.295887%


# Model training

In [7]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU count: 1
Current device index: 0
Current device name: NVIDIA GeForce RTX 4080 SUPER


In [8]:
import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast

# 0. 벤치마크 켜기 (속도 향상)
torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler('cuda')

epochs = 15
save_path = "best_model.pth"
best_val_loss = float("inf")

criterion = nn.CrossEntropyLoss(label_smoothing=0.0).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            output = model(x)
            loss = criterion(output, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        # [중복 코드 삭제됨]

        # 통계 (AMP로 계산된 output을 그대로 사용)
        train_loss_sum += loss.item() * y.size(0)
        
        # 예측값 계산 (여기는 그라디언트 필요 없으므로 detach 추천)
        preds = torch.argmax(output.detach(), dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # VALIDATION (그대로 유지)
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # 검증 때는 AMP를 굳이 안 써도 되지만, 쓰면 조금 더 빠를 수 있음
            with autocast():
                output = model(x)
                val_loss_batch = criterion(output, y)

            val_loss_sum += val_loss_batch.item() * y.size(0)
            preds = torch.argmax(output, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    scheduler.step()

    # MODEL SAVE
    save_dict = {
        "epoch": epoch,
        "model_state_dict": (
            model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict()
        ),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    torch.save(save_dict, f'epoch_{epoch}.pth')

    # BEST MODEL SPECIFICATION
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        save_dict = {
            "epoch": epoch,
            "model_state_dict": (
                model.module.state_dict() if isinstance(model, nn.DataParallel)
                else model.state_dict()
            ),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
        }

        torch.save(save_dict, save_path)
        print(f"Best model saved at epoch {epoch} (val_loss={val_loss:.4f})")

Epoch 0 [Val]:   0%|          | 0/71 [00:00<?, ?it/s]C:\Users\MAIN\AppData\Local\Temp\ipykernel_20008\2364548934.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.82it/s]


Epoch 00 | Train Loss: 2.4999 | Train Acc: 0.1916 | Val Loss: 2.3213 | Val Acc: 0.2566
Best model saved at epoch 0 (val_loss=2.3213)


Epoch 1 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.81it/s]


Epoch 01 | Train Loss: 2.1419 | Train Acc: 0.3085 | Val Loss: 2.0295 | Val Acc: 0.3462
Best model saved at epoch 1 (val_loss=2.0295)


Epoch 2 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.77it/s]


...

Epoch 12 | Train Loss: 0.9520 | Train Acc: 0.6841 | Val Loss: 1.1872 | Val Acc: 0.6180
Best model saved at epoch 12 (val_loss=1.1872)


Epoch 13 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.72it/s]


Epoch 13 | Train Loss: 0.9108 | Train Acc: 0.6969 | Val Loss: 1.1888 | Val Acc: 0.6174


Epoch 14 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.75it/s]


Epoch 14 | Train Loss: 0.8936 | Train Acc: 0.7023 | Val Loss: 1.1788 | Val Acc: 0.6214
Best model saved at epoch 14 (val_loss=1.1788)


In [9]:
'''
# 모델 불러오기
checkpoint = torch.load("best_model.pth", map_location=device)

# 먼저 순수 모델을 만들고 로드
base_model = ConvNeXtBN(num_classes=15)
base_model.load_state_dict(checkpoint["model_state_dict"])

# 그 다음에 DataParallel로 감쌈
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
'''

'\n# 모델 불러오기\ncheckpoint = torch.load("best_model.pth", map_location=device)\n\n# 먼저 순수 모델을 만들고 로드\nbase_model = ConvNeXtBN(num_classes=15)\nbase_model.load_state_dict(checkpoint["model_state_dict"])\n\n# 그 다음에 DataParallel로 감쌈\nif torch.cuda.device_count() > 1:\n    model = torch.nn.DataParallel(base_model)\nelse:\n    model = base_model\n\nmodel = model.to(device)\n\noptimizer = torch.optim.Adam(model.parameters(), lr=0.001)\noptimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n'

In [10]:
'''
# 모델 로드 검증
missing_keys, unexpected_keys = base_model.load_state_dict(
    checkpoint["model_state_dict"], strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

# 파라미터 값 확인
with torch.no_grad():
    w = base_model.downsample_layers[0][0].weight

print("Sample weight stats:")
print("  mean:", w.mean().item())
print("  std :", w.std().item())
print("  min :", w.min().item())
print("  max :", w.max().item())

# forward 테스트
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out = model(dummy)

print("Output shape:", out.shape)
print("Has NaN:", torch.isnan(out).any().item())
'''

'\n# 모델 로드 검증\nmissing_keys, unexpected_keys = base_model.load_state_dict(\n    checkpoint["model_state_dict"], strict=False\n)\n\nprint("Missing keys:", missing_keys)\nprint("Unexpected keys:", unexpected_keys)\n\n# 파라미터 값 확인\nwith torch.no_grad():\n    w = base_model.downsample_layers[0][0].weight\n\nprint("Sample weight stats:")\nprint("  mean:", w.mean().item())\nprint("  std :", w.std().item())\nprint("  min :", w.min().item())\nprint("  max :", w.max().item())\n\n# forward 테스트\nmodel.eval()\nwith torch.no_grad():\n    dummy = torch.randn(2, 3, 224, 224).to(device)\n    out = model(dummy)\n\nprint("Output shape:", out.shape)\nprint("Has NaN:", torch.isnan(out).any().item())\n'

In [11]:
'''
print("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))
print("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))
print("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))
'''

'\nprint("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))\nprint("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))\nprint("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))\n'

# Submit
Do not edit the submission code below.

In [12]:
submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 49295887
Parameter usage : 49.295887%


100%|██████████| 59/59 [00:14<00:00,  4.02it/s]
